In [1]:
import secrets

# Large prime for the finite field (secret must be < PRIME)
PRIME = 2**127 - 1  # Mersenne prime, big enough for most demos


def random_coeffs(k_minus_1, prime=PRIME):
    """Generate k-1 random coefficients in Z_p."""
    return [secrets.randbelow(prime) for _ in range(k_minus_1)]


def eval_poly(x, coeffs, prime=PRIME):
    """Evaluate polynomial f(x) with given coeffs over Z_p.
       coeffs[0] = a0, coeffs[1] = a1, ..."""
    result = 0
    for power, a in enumerate(coeffs):
        result = (result + a * pow(x, power, prime)) % prime
    return result


def split_secret(secret, k, n, prime=PRIME):
    """
    Create n shares from secret with threshold k using Shamir's scheme.
    Returns list of (x, y) shares.
    """
    assert 0 <= secret < prime, "Secret must be in field range"

    # Polynomial: f(x) = a0 + a1 x + ... + a_{k-1} x^{k-1}, where a0 = secret
    coeffs = [secret] + random_coeffs(k - 1, prime)

    shares = []
    for x in range(1, n + 1):
        y = eval_poly(x, coeffs, prime)
        shares.append((x, y))
    return shares


def modinv(a, prime=PRIME):
    """Modular inverse using extended Euclidean algorithm."""
    if a == 0:
        raise ZeroDivisionError("No inverse for 0")
    lm, hm = 1, 0
    low, high = a % prime, prime
    while low > 1:
        r = high // low
        nm, new = hm - lm * r, high - low * r
        lm, low, hm, high = nm, new, lm, low
    return lm % prime


def lagrange_interpolate_zero(shares, prime=PRIME):
    """
    Lagrange interpolation at x = 0 to recover f(0) (the secret).
    shares: list of (x, y)
    """
    secret = 0
    for j, (xj, yj) in enumerate(shares):
        num, den = 1, 1
        for m, (xm, _) in enumerate(shares):
            if m == j:
                continue
            num = (num * (-xm)) % prime         # (0 - xm)
            den = (den * (xj - xm)) % prime
        lj = num * modinv(den, prime) % prime   # Lagrange basis at 0
        secret = (secret + yj * lj) % prime
    return secret


def reconstruct_secret(shares_subset, prime=PRIME):
    """Reconstruct the secret from any k shares."""
    return lagrange_interpolate_zero(shares_subset, prime)


if __name__ == "__main__":
    # Example: (k, n) = (3, 5)
    secret = 1234
    k, n = 3, 5

    # 1) Dealer creates shares
    shares = split_secret(secret, k, n)
    print("All shares:")
    for s in shares:
        print(s)

    # 2) Pick ANY k shares to reconstruct
    subset = shares[:3]  # e.g., first 3 shares
    recovered = reconstruct_secret(subset)

    print("\nUsing shares:", subset)
    print("Recovered secret:", recovered)
    print("Success:", recovered == secret)



All shares:
(1, 114019642678005063205030341171375056979)
(2, 16383993414180857600396047380716812787)
(3, 47375419129465846649471726059793480112)
(4, 36852736363390798620570073492720953227)
(5, 154957128576424945245378393395383337859)

Using shares: [(1, 114019642678005063205030341171375056979), (2, 16383993414180857600396047380716812787), (3, 47375419129465846649471726059793480112)]
Recovered secret: 1234
Success: True
